Removing amendments from project, but keeping code just in case

In [ ]:
# Packages & functions

import requests
import random
import json
import time
from requests.exceptions import HTTPError
from pathlib import Path
from datetime import datetime, timezone
import os
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor, as_completed
import psycopg2
import pytz
import re


CURRENT_YEAR = 2026

load_dotenv()  # loads variables from .env

# Base url for extraction
base_url = "https://api.congress.gov/v3"
API_KEY = os.getenv("API_KEY")

# API call
session = requests.Session()

# API function
def get_api_info(search_string, max_retries=5):
    url = f"{base_url}/{search_string}&api_key={API_KEY}"
    for i in range(max_retries):
        response = session.get(url)
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 429:
            wait = (2 ** i) + random.uniform(0, 0.5)  # exponential backoff + jitter
            print(f"429 Rate limited. Waiting {wait:.1f}s...")
            time.sleep(wait)
        else:
            response.raise_for_status()
    raise Exception(f"Failed after {max_retries} retries: {url}")

# Save data as json file
def save_to_file(data, file_path):
    file_path = Path(file_path)
    file_path.parent.mkdir(parents=True, exist_ok=True)

    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

output_path = "../data/silver"

retrieved_at = datetime.now(timezone.utc).isoformat()

# Importing raw files from silver
def import_silver(file_name):
    with open(f"../data/silver/{file_name}", "r") as f:
        raw_data = json.load(f)
    return raw_data

# Exporting clean files to gold
def export_gold(file_name, clean_data, indent_num):
    with open(f"../data/gold/{file_name}", "w") as f:
        json.dump(clean_data, f, indent=indent_num, ensure_ascii=False)
    print(f"Merge complete. Output saved to ../data/gold/{file_name}")

# Set the type of each variable
def cast_record(record, type_map):
    for field, field_type in type_map.items():
        value = record.get(field)

        if value in (None, ""):
            record[field] = None
        else:
            try:
                record[field] = field_type(value)
            except (ValueError, TypeError):
                record[field] = None

    return record

In [ ]:
# Amendment data (JSON via API)

# Amendment data for 119th Congress (records = 4455 as of 2/9/26)
pages = []

for offset in range(0, 4455, 250):
    search_string = f"amendment/119?format=json&offset={offset}&limit=250"
    page_data = get_api_info(search_string)
    pages.append(page_data)

final_payload = {
    "source": "https://api.congress.gov/",
    "retrieved_at": retrieved_at,
    "congress" : 119,
    "pages": pages
}

save_to_file(final_payload, f"{output_path}/amendments_119.json")


In [ ]:
# CLEAN AMENDMENT DATA

data = import_silver("amendments_119.json")

cleaned_amendments = []

for page in data.get("pages", []):
    for amendment in page.get("amendments", []):

        congress = amendment.get("congress")
        amendment_type = amendment.get("type")
        number = amendment.get("number")

        # Build amendment_id
        amendment_id = f"{amendment_type}{number}_{congress}"

        # Determine chamber
        if amendment_type == "SAMDT":
            chamber = "S"
        elif amendment_type == "HAMDT":
            chamber = "H"
        else:
            chamber = None

        # Prefer description, fallback to purpose
        purpose = amendment.get("description") or amendment.get("purpose")

        cleaned_amendments.append({
            "amendment_id": amendment_id,
            "amendment_type": amendment_type,
            "congress" : congress,
            "chamber" : chamber,
            "purpose": purpose
        })

# Make sure data type is correct
TYPE_MAP_AMEDMENTS = {
    "amendment_id": str,
    "amendment_type": str,
    "congress": int,
    "chamber": str,
    "purpose": str,
}

cleaned_amendments = [cast_record(r, TYPE_MAP_AMEDMENTS) for r in cleaned_amendments]

# Check data:
print("Total records:", len(cleaned_amendments))
print("Unique IDs:", len(set(r["amendment_id"] for r in cleaned_amendments)))

export_gold("amendments_119.json", cleaned_amendments, 2)



In [ ]:
# Importing amendment data to database

# Connect to db
conn = psycopg2.connect(
    dbname="congress_db",
    user="postgres",
    password="data4001",
    host="localhost",
    port="5432"
)

cur = conn.cursor()
conn.commit()

# Import amendments
with open("../data/gold/amendments_119.json", "r") as f:
    amendments = json.load(f)

amendment_insert_query = """
INSERT INTO amendments (
    amendment_id,
    amendment_type,
    congress,
    chamber,
    purpose
)
VALUES (%s, %s, %s, %s, %s)
ON CONFLICT (amendment_id) DO NOTHING;
"""

for a in amendments:
    cur.execute(amendment_insert_query, (
        a.get("amendment_id"),
        a.get("amendment_type"),
        a.get("congress"),
        a.get("chamber"),
        a.get("purpose")
    ))

conn.commit()